
# Cantera Equilibrium Skeleton (TP, VCS Minimization)

This notebook is a minimal, self-contained scaffold showing how we set up and run
thermodynamic equilibrium calculations in Cantera for aqueous systems (e.g., Titan
impact-pond analogs). It is intended for readers and reviewers who want to understand
the exact simulation flow and reproduce results with their own species YAML files.

**What this does**
- Loads an aqueous phase from a provided YAML (with NASA-9 polynomials).
- Defines pressure, a temperature list (in °C), and an initial composition.
- Normalizes the composition, sets the `TPX` state, and equilibrates using the VCS solver.
- Records final mole fractions for all species vs. temperature to a CSV file.

**What you must provide**
- A Cantera YAML file containing the relevant species and an `aqueous` phase definition.
- The folder path containing your YAML(s). If your YAMLs depend on relative includes,
  place them in the same directory and call `ct.add_directory(...)` accordingly.

**Assumptions**
- Thermodynamic data are represented as NASA-9 polynomials fitted from a database
  (e.g., CHNOSZ-derived fits). The equilibrium is obtained by **Gibbs energy minimization**
  using Cantera's `VCS` algorithm under constant temperature and pressure (`"TP"`).

> **Tip:** This is a skeleton. Start here, then expand (e.g., NH3 sweeps, multiple YAMLs,
> alternate pressures, additional outputs).


In [ ]:

# =============================================================
# Minimal Cantera Equilibrium Skeleton
# =============================================================
# Purpose: demonstrate the essential steps to (1) load a phase,
# (2) define state & composition, (3) equilibrate with VCS at TP,
# (4) export species mole fractions vs. temperature.
#
# Expand as needed for your study (parameter sweeps, plots, etc.).
# -------------------------------------------------------------

from pathlib import Path
import cantera as ct
import pandas as pd

# ---------------- User Configuration (EDIT THESE) ----------------
# Path that contains your YAML files (and any includes they need).
ROOT_DIR   = Path.cwd()  # e.g., Path('Input files/with_NH3')
PHASE_FILE = 'filename.yaml'   # <-- replace with your YAML filename
PHASE_NAME = 'aqueous'              # <-- replace if your phase name differs

PRESSURE_PA = 1.5e5                 # Example: 1.5 bar in Pascals
TEMPS_C     = [0, 25, 50, 75, 100]  # Temperatures to evaluate (°C)

# Initial amounts. Use any convenient totals;
# they will be normalized to mole fractions automatically.
BASE_COMP = {
    'H2O(l)'  : 1.00,
    'HCN(aq)' : 0.02,
    'C2H2(aq)': 0.04,
    # 'NH3(aq)': 0.05,  # Uncomment/edit to include ammonia, etc.
    # Add/remove species keys to match your YAML species names.
}

OUTPUT_CSV = 'equilibrium_results_skeleton.csv'  # saved in notebook folder
# ----------------------------------------------------------------

# Make the YAML directory discoverable (important for relative includes)
ct.add_directory(str(ROOT_DIR))

# Load the aqueous phase
sol = ct.Solution(PHASE_FILE, PHASE_NAME)

# Helper: convert unnormalized amounts to mole fractions
def normalize_amounts(amounts: dict) -> dict:
    total = sum(amounts.values())
    if total <= 0:
        raise ValueError("Total initial amount must be positive.")
    return {k: v/total for k, v in amounts.items()}

# Prepare output table columns: Temperature + all species in phase
columns = ['Temperature_C'] + list(sol.species_names)
rows = []

# Loop over temperatures and run TP equilibrium (VCS)
for T_C in TEMPS_C:
    T_K = T_C + 273.15
    # Fresh solution object each iteration (keeps each state independent)
    aq = ct.Solution(PHASE_FILE, PHASE_NAME)

    # Normalize composition and set state
    comp_X = normalize_amounts(BASE_COMP)
    aq.TPX = T_K, PRESSURE_PA, comp_X

    # Equilibrate using VCS (robust for multi-component systems)
    aq.equilibrate('TP', solver='vcs', max_steps=100000)

    # Collect mole fractions for every species at this temperature
    row = [T_C] + [aq.X[aq.species_index(sp)] for sp in aq.species_names]
    rows.append(row)

# Save to CSV for provenance/reuse
df = pd.DataFrame(rows, columns=columns)
df.to_csv(OUTPUT_CSV, index=False)

print(f"  Done. Wrote {len(df)} rows × {len(df.columns)} columns to {OUTPUT_CSV}")
print("   Phase file:", PHASE_FILE, "| Phase name:", PHASE_NAME)
print("   Temperatures (°C):", TEMPS_C)

# ---------------- Optional: quick look (uncomment to preview) ----------------
# import matplotlib.pyplot as plt
# plt.figure(figsize=(9,5))
# for sp in sol.species_names:
#     plt.plot(df['Temperature_C'], df[sp], label=sp)
# plt.yscale('log')
# plt.xlabel('Temperature (°C)')
# plt.ylabel('Equilibrium mole fraction (log scale)')
# plt.title('Equilibrium composition vs. Temperature (skeleton run)')
# plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
# plt.tight_layout()
# plt.show()
